# Magic Methods in Python — Explained Like You Are 5

## Goal

**Magic methods** are special methods that tell Python how our objects should behave with familiar built-in tools.

Imagine making a new toy. Python asks questions such as:

- What should I show when someone prints this toy?
- How long is this toy box?
- What should happen when someone asks for item number 2?
- Can two toys be compared?

Magic methods let the class answer those questions.

> **Big idea:** Magic methods connect your objects to ordinary Python syntax such as `print()`, `len()`, indexing, comparisons, loops, and operators.

## Why are they called dunder methods?

**Dunder** means **double underscore**. These methods begin and end with two underscores:

```text
__init__   __str__   __len__
```

They are not supernatural. Python automatically calls them behind the scenes.

```python
print(book)       # Python uses book.__str__()
len(playlist)     # Python uses playlist.__len__()
playlist[0]       # Python uses playlist.__getitem__(0)
```

Normally, use the friendly syntax on the left instead of directly calling the dunder method on the right.

## Quick map of common magic methods

| Magic method | Python syntax | Job |
|---|---|---|
| `__init__` | `Thing(...)` | Set up a new object |
| `__str__` | `print(obj)` or `str(obj)` | Friendly text for people |
| `__repr__` | `repr(obj)` | Detailed text for developers |
| `__len__` | `len(obj)` | Return the object's length |
| `__getitem__` | `obj[index]` | Read an item |
| `__setitem__` | `obj[index] = value` | Change an item |
| `__contains__` | `item in obj` | Check membership |
| `__iter__` | `for item in obj` | Allow looping |
| `__call__` | `obj()` | Make an object callable |
| `__eq__` | `obj1 == obj2` | Check equality |
| `__bool__` | `bool(obj)` | Decide truth or falsehood |

## 1. Looking at an ordinary object — original example

Even an empty class receives many magic methods from Python's base `object` class. `dir(person)` lists everything the object knows.

Most names beginning with `__` are built-in tools supplied by Python.

In [1]:
class Person:
    pass


person = Person()
dir(person)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__']

### Printing before `__str__`

Without a custom text method, Python shows the class name and a memory-related identifier. It is correct, but it is not friendly to read.

In [2]:
print(person)

## 2. `__init__`: setting up an object — original example

Python automatically calls `__init__` after creating a new object. It is commonly called the **constructor**, although technically `__new__` creates the object and `__init__` initializes it.

This version stores a name and age, but printing still uses Python's default representation.

In [3]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age


person = Person("Krish", 34)
print(person)

## 3. `__str__` and `__repr__` — original example

These two methods give an object readable text:

- `__str__` is a friendly name tag for users.
- `__repr__` is a detailed label for programmers and debugging.

A useful `repr` often looks like valid Python code that could recreate the object.

In [4]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __str__(self):
        return f"{self.name}, {self.age} years old"

    def __repr__(self):
        return f"Person(name={self.name!r}, age={self.age!r})"


person = Person("Krish", 34)
print(person)
print(repr(person))

Krish, 34 years old
Person(name='Krish', age=34)


### `str()` versus `repr()`

`print(person)` uses `__str__`. A list displays its items using each item's `__repr__`, which is why a good `repr` is helpful during debugging.

In [5]:
people = [Person("Krish", 34), Person("Riya", 12)]

print("Friendly str:", str(people[0]))
print("Developer repr:", repr(people[0]))
print("Inside a list:", people)

Friendly str: Krish, 34 years old
Developer repr: Person(name='Krish', age=34)
Inside a list: [Person(name='Krish', age=34), Person(name='Riya', age=12)]


## 4. `__len__`: teaching `len()`

A playlist object can decide that its length means **the number of songs**.

`__len__` must return a non-negative integer.

In [6]:
class Playlist:
    def __init__(self, name, songs):
        self.name = name
        self.songs = list(songs)

    def __len__(self):
        return len(self.songs)


playlist = Playlist("Happy Songs", ["Sunrise", "Smile", "Dance"])
print("Number of songs:", len(playlist))

Number of songs: 3


## 5. `__getitem__` and `__setitem__`: square brackets

These methods make a custom object feel like a list:

- `playlist[0]` calls `__getitem__(0)`.
- `playlist[1] = "New Song"` calls `__setitem__(1, "New Song")`.

Delegating to the inner list also gives us normal list behavior such as negative indexes and `IndexError`.

In [7]:
class EditablePlaylist(Playlist):
    def __getitem__(self, index):
        return self.songs[index]

    def __setitem__(self, index, value):
        self.songs[index] = value


editable = EditablePlaylist("Road Trip", ["Song A", "Song B", "Song C"])
print("First song:", editable[0])

editable[1] = "Favorite Song"
print("Updated second song:", editable[1])
print("Last song:", editable[-1])

First song: Song A
Updated second song: Favorite Song
Last song: Song C


## 6. `__contains__` and `__iter__`: membership and loops

- `"Song A" in playlist` uses `__contains__`.
- `for song in playlist` uses `__iter__`.

We return an iterator for the internal song list, so Python can visit each song one at a time.

In [8]:
class LoopablePlaylist(EditablePlaylist):
    def __contains__(self, song):
        return song in self.songs

    def __iter__(self):
        return iter(self.songs)


loopable = LoopablePlaylist("Study", ["Focus", "Calm", "Finish"])
print("Does it contain Calm?", "Calm" in loopable)

for song in loopable:
    print("Playing:", song)

Does it contain Calm? True
Playing: Focus
Playing: Calm
Playing: Finish


## 7. `__call__`: using an object like a function

If a class defines `__call__`, its objects can be followed by parentheses.

The counter below remembers its value between calls, something an ordinary simple function may not do by itself.

In [9]:
class Counter:
    def __init__(self):
        self.count = 0

    def __call__(self):
        self.count += 1
        return self.count


counter = Counter()
print(counter())
print(counter())
print(counter())

1
2
3


## 8. `__eq__`: deciding when objects are equal

Without a custom `__eq__`, two separate objects are usually unequal even if they store identical values.

This method says two students are equal when their student IDs match. Returning `NotImplemented` lets Python handle unsupported types correctly.

In [10]:
class Student:
    def __init__(self, student_id, name):
        self.student_id = student_id
        self.name = name

    def __eq__(self, other):
        if not isinstance(other, Student):
            return NotImplemented
        return self.student_id == other.student_id


first = Student(101, "Aarav")
second = Student(101, "Aarav Kumar")
third = Student(202, "Riya")

print(first == second)
print(first == third)
print(first == 101)

True
False
False


## 9. `__bool__`: truth and falsehood

`bool(obj)` and `if obj:` use `__bool__`. A shopping cart can decide it is `False` when empty and `True` when it contains items.

If `__bool__` is missing, Python may use `__len__`; an object with length zero is considered false.

In [11]:
class ShoppingCart:
    def __init__(self, items=None):
        self.items = list(items or [])

    def __bool__(self):
        return bool(self.items)


empty_cart = ShoppingCart()
full_cart = ShoppingCart(["book"])

print("Empty cart:", bool(empty_cart))
print("Full cart:", bool(full_cart))

Empty cart: False
Full cart: True


## 10. Context-manager magic: `__enter__` and `__exit__`

A `with` statement asks an object to set something up and clean it afterward:

- `__enter__` runs at the beginning.
- `__exit__` runs at the end, even when the block has a problem.

Files use this pattern so they are closed safely.

In [12]:
class ToyBox:
    def __enter__(self):
        print("Opening the toy box")
        return self

    def play(self):
        print("Playing with a toy")

    def __exit__(self, error_type, error, traceback):
        print("Closing the toy box")
        return False


with ToyBox() as toy_box:
    toy_box.play()

Opening the toy box
Playing with a toy
Closing the toy box


## Lifecycle note: `__new__` and `__del__`

- `__new__` creates the object before `__init__` sets it up. Most classes do not need to override it.
- `__del__` may run when an object is being destroyed, but its timing is not dependable. Do not rely on it for important cleanup; prefer context managers.

## Common mistakes

1. Forgetting that methods must return the correct type: `__len__` needs an integer and `__str__` needs a string.
2. Calling dunder methods directly instead of using `len(obj)`, `str(obj)`, or `obj[index]`.
3. Making `__repr__` vague, which makes debugging difficult.
4. Returning `False` instead of `NotImplemented` for unsupported comparison types.
5. Adding surprising behavior that users would not expect from normal Python syntax.

## Easy revision cheat sheet

| When you write... | Python tries... | Remember... |
|---|---|---|
| `Thing(...)` | `__new__`, then `__init__` | Create, then set up |
| `print(obj)` | `__str__` | Friendly text |
| `repr(obj)` | `__repr__` | Developer text |
| `len(obj)` | `__len__` | Return a non-negative integer |
| `obj[key]` | `__getitem__` | Read an item |
| `obj[key] = value` | `__setitem__` | Change an item |
| `item in obj` | `__contains__` | Membership test |
| `for item in obj` | `__iter__` | Return an iterator |
| `obj()` | `__call__` | Object behaves like a function |
| `obj1 == obj2` | `__eq__` | Define equality |
| `if obj:` | `__bool__` or `__len__` | Decide truthiness |
| `with obj:` | `__enter__`, `__exit__` | Safe setup and cleanup |

### Five-second revision

**Dunder methods are hooks that let custom objects understand normal Python syntax.**